# Meshing the native BREP

`gbs.brep.mesh` triangulates a shape: watertight along the shared edges, within a **deflection** (distance between the mesh and the geometry) and an **angle** (turn of the normals across a triangle). Each face is triangulated in its parameter space, scaled by the spacing its curvature asks for in each direction: the triangles of a cylinder run long along its generatrices, those of a sphere are isotropic.

In [ ]:
import math
import pyvista as pv
import gbs_examples as ex

brep = ex.brep
try:
    pv.set_jupyter_backend("trame")  # interactive
except Exception:
    pv.set_jupyter_backend("static")  # images only

def show_mesh(m, shape, options, title=""):
    poly, mesh = ex.face_polydata(m, shape, options)
    p = pv.Plotter()
    p.add_mesh(poly, color="#9ecae1", show_edges=True, edge_color="#3a5f7d", line_width=0.5)
    lines = ex.edge_lines(m, shape)
    if lines is not None:
        p.add_mesh(lines, color="#222222", line_width=2)
    p.add_text(f"{title}{len(mesh.triangles)} triangles, {mesh.open_sides()} open sides, volume {mesh.volume():.5g}", font_size=9)
    p.show()
    return mesh

## The deflection

The volume of the mesh converges to that of the sphere (4/3 pi r^3 = 4.18879 for r = 1) as the deflection decreases.

In [ ]:
m = brep.Model()
s = ex.sphere(m, 1.0)
for deflection in (1e-1, 1e-2, 1e-3):
    mesh = brep.mesh(m, s, brep.MeshOptions(deflection=deflection))
    print(f"deflection {deflection:g}: {len(mesh.triangles)} triangles, volume {mesh.volume():.6f} (exact {4 / 3 * math.pi:.6f})")
show_mesh(m, s, brep.MeshOptions(deflection=1e-2), "sphere: ")

## The angle, the smallest angle

`angle` bounds the turn of the normals across a triangle; `min_angle` (20 degrees by default, 0 for none) refines the triangles under it in the scaled parameter space.

In [ ]:
m = brep.Model()
c = ex.cone(m, 1.0, 2.0)
show_mesh(m, c, brep.MeshOptions(deflection=1e-2), "cone, smallest angle 20 deg: ")
show_mesh(m, c, brep.MeshOptions(deflection=1e-2, min_angle=0.0), "cone, no smallest angle: ")

## A boolean result

The trimmed faces of a boolean mesh like the others; the mesh of the solid is closed.

In [ ]:
m = brep.Model()
r = brep.cut(m, ex.box(m, (0, 0, 0), (2, 3, 4)), ex.sphere(m, 1.5, (2, 3, 4), (1, 1, 1)))
mesh = show_mesh(m, r.shape, brep.MeshOptions(deflection=1e-3), "box minus sphere: ")

## Export

STL (binary or ASCII) and OBJ (a normal per corner, a group per face).

In [ ]:
brep.write_stl(mesh, "box_minus_sphere.stl")
brep.write_obj(mesh, "box_minus_sphere.obj")